You have at your disposal 100000 images of human faces, and their occlusion label.
The goal of this challenge is to regress the percentage of the face that is occluded.
We also want to have similar performances on female and male, the gender label is given for the train database

Below is the formula of the evaluation score

$$
 Err = \frac{\sum_{i}{w_i(p_i - GT_i)^2}}{\sum_{i}{w_i}}, w_i = \frac{1}{30} + GT_i
$$

$$
Score = \frac{Err_F + Err_M}{2} + \left | Err_F - Err_M \right |
$$

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from collections import OrderedDict

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

import cv2
import os
import json

### Load dataframes

In [ ]:
df_test = pd.read_csv("../../occlusion_datasets/test_students.csv", delimiter=',')

image_dir = "../../crops/Crop_224_5fp_100K"

In [ ]:
df_test.head()

#### Remove nan values

In [ ]:
df_test = df_test.dropna()
df_test["row_id"] = range(len(df_test))

### Split Dataframe in test1 and test2

In [ ]:
split_idx = len(df_test) // 2
df_test1 = df_test.iloc[:split_idx].reset_index()
df_test2 = df_test.iloc[split_idx:].reset_index()

In [ ]:
len(df_test1), len(df_test2), len(df_test)

### Check that all images are read correctly

In [ ]:
for idx, row in tqdm(df_test1.iterrows(), total=len(df_test1)):
    try:
        filename = df_test1.loc[idx, 'filename']
        img2display = Image.open(f"{image_dir}/{filename}")
    except ValueError as e:
        print(idx, e)

### Make Dataset and Dataloader

In [ ]:
class Dataset(torch.utils.data.Dataset):
    'Characterizes a dataset for PyTorch'
    def __init__(self, df, image_dir, training=True):
         'Initialization'
         self.training = training
         self.image_dir = image_dir
         self.df = df
         self.transform = transforms.ToTensor()
         
    def __len__(self):
        'Denotes the total number of samples'
        return len(self.df)

    def __getitem__(self, index):
        'Generates one sample of data'
        # Select sample
        row = self.df.loc[index]
        filename = row['filename']

        # Load data and get label
        img = Image.open(f"{image_dir}/{filename}")

        X = cv2.imread(f"{image_dir}/{filename}")

        if self.training:
            y = row['FaceOcclusion']
            y = np.float32(y)
            gender = row['gender']
            return X, y, gender, filename
        else:
            y = row['row_id']
            gender = None
            return X, y, filename

# Training-Free track

## Installation

In [ ]:
from pathlib import Path
import subprocess
import sys

import gdown

### 3DDFA-V2

In [ ]:
repo_dir = Path("3DDFA_V2")

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/cleardusk/3DDFA_V2.git"],
        check=True
    )

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", "3DDFA_V2/requirements.txt"],
        check=True
    )

    print("Repository cloné et dépendances installées.")
else:
    print("Repository déjà présent, installation ignorée.")

### BiSeNet

In [ ]:
repo_dir = Path("face-parsing.PyTorch")

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/zllrunning/face-parsing.PyTorch.git"],
        check=True
    )

    print("Repository cloné.")
else:
    print("Repository déjà présent, installation ignorée.")

Le modèle pré-entrainé n'est pas directement accessible depuis le repo Git, mais depuis un Drive. Voici les informations pour le récupérer en cas d'échec du téléchargement automatique.

Lien pour la copie des poids : https://drive.google.com/open?id=154JgKpzCPW82qINcVieuPH3fZ2e0P812
A placer dans un sous-dossier weights, à partir du dossier courant du notebook

In [ ]:
Path("weights").mkdir(exist_ok=True)

file_id = "154JgKpzCPW82qINcVieuPH3fZ2e0P812"
output = "weights/79999_iter.pth"

if not Path(output).exists():
    gdown.download(
        f"https://drive.google.com/uc?id={file_id}",
        output,
        quiet=False
    )
    print("Modèle chargé")
else:
    print("Modèle déjà installé")

## Model Imports

In [ ]:
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device ='cpu'
print(device)

In [ ]:
#Face Analysis

from insightface.app import FaceAnalysis

app = FaceAnalysis(
    name='buffalo_l',
    providers=[
        'CPUExecutionProvider'
    ]
)
app.prepare(
    ctx_id=0,
    det_size=(224, 224)
)

In [ ]:
# 3DDFA-V2

ROOT = Path.cwd()
THREEDDFA_PATH = ROOT / "3DDFA_V2"
sys.path.append(str(THREEDDFA_PATH))
print(THREEDDFA_PATH)

from TDDFA import TDDFA

MODEL_PATH = Path.cwd() / "3DDFA_V2" / "weights" / "mb1_120x120.pth"

cfg = {
    'arch': 'mobilenet',
    'checkpoint_fp': str(MODEL_PATH),
    'gpu_mode': False
}

tddfa = TDDFA(**cfg)

from utils.pose import viz_pose

In [ ]:
#BiSeNet

BISENET_PATH = ROOT / "face-parsing.PyTorch"
sys.path.append(str(BISENET_PATH))

from model import BiSeNet
import torch

WEIGHTS = (
    Path.cwd()
    / "weights"
    / "79999_iter.pth"
)

assert WEIGHTS.exists()

n_classes = 19
net = BiSeNet(n_classes=n_classes)
net.load_state_dict(
    torch.load(
        WEIGHTS,
        map_location="cpu"
    )
)

net.eval().to(device)

In [ ]:
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

seg_processor = SegformerImageProcessor.from_pretrained("jonathandinu/face-parsing")
seg_model = SegformerForSemanticSegmentation.from_pretrained("jonathandinu/face-parsing")
seg_model.to(device)

## Pipeline and configuration

In [ ]:
from torchvision import transforms

# Segmentation classes for BiSeNet and SegFormer

VISIBLE_FACE_CLASSES_B = [
    0,   # background (in case of missed detection)
    1,   # skin
    2,   # left eyebrow
    3,   # right eyebrow
    4,   # left eye
    5,   # right eye
    6,   # eyeglasses
    10,  # nose
    11,  # mouth
    12,  # upper lip
    13  # lower lip
]

VISIBLE_FACE_CLASSES_S = [
    1,   # skin
    2,  # nose
    3,   # eyeglasses
    4,   # left eye
    5,   # right eye
    6,   # left eyebrow
    7,   # right eyebrow
    10,  # mouth
    11,  # upper lip
    12  # lower lip
]

HAIR_B = [17]
HAT_B = [18]
HAIR_S = [13]
HAT_S = [14]

# === Constantes pour la detection des erreurs de segmentation (stratégie fallback) ===
SKIN_ONLY_B = [1, 2, 3, 4, 5, 7, 8, 10, 11, 12, 13]
SKIN_ONLY_S = [1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12]
BG_B        = [0]

BI_ERROR_THRESHOLD = 0.7
SF_ERROR_THRESHOLD = 0.3

# Loading weights pre-computed
with open("weights_base.json", "r", encoding="utf-8") as f:
    weights = json.load(f)

F_WEIGHTS = weights["female"]
M_WEIGHTS = weights["male"]

# Transformation in tensor
to_tensor = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    ),
])

In [ ]:
def overlay_mask(image,mask,color,alpha=0.4):
    '''
    Superimposing semgented mask on initial image for visualization
    '''
    overlay = image.copy()
    overlay[mask > 0] = color
    blended = cv2.addWeighted(
        overlay,
        alpha,
        image,
        1 - alpha,
        0
    )
    return blended

In [ ]:
# === Sous-fonction 1 : detection visage + genre (InsightFace) ===
def detect_face_and_gender(app, img):
    """Detecte le visage principal dans l'image avec InsightFace.

    Args:
        app: instance FaceAnalysis InsightFace
        img: numpy array BGR

    Returns:
        (face_bbox, pred_gender) ou (None, None) si aucun visage detecte.
        pred_gender est 'F' ou 'M' (sortie face.sex).
    """
    faces = app.get(img)
    if not faces:
        return None, None
    face = faces[0]
    return face.bbox, face.sex


In [ ]:
# === Sous-fonction 2 : calcul du mask theorique (3DDFA-V2 + correction Julien) ===
def compute_theoretical_face_mask(tddfa, img, bbox):
    """Calcule le mask theorique du visage par reconstruction 3D et applique la
    correction empirique de Julien (scale_x=0.9, scale_y=1.05, tx=15, ty=-10).

    Args:
        tddfa: instance TDDFA
        img: numpy array BGR
        bbox: face bbox (sortie InsightFace)

    Returns:
        (mask_theoretical, total_pixels) : matrice HxW uint8 et nombre total de pixels du mask.
    """
    # Reconstruction 3D du visage
    param_lst, roi_box_lst = tddfa(img, [bbox])
    ver_lst = tddfa.recon_vers(param_lst, roi_box_lst, dense_flag=True)

    # Convex hull des vertices projetes -> mask 2D
    mask_theoretical = np.zeros(img.shape[:2], dtype=np.uint8)
    pts = ver_lst[0][:2, :].T.astype(np.int32)
    cv2.fillConvexPoly(mask_theoretical, cv2.convexHull(pts), 1)

    # Correction empirique de Julien
    scale_x = 0.9
    scale_y = 1.05
    tx = 15
    ty = -10
    M = np.array([[scale_x, 0, tx], [0, scale_y, ty]], dtype=np.float32)
    mask_theoretical = cv2.warpAffine(mask_theoretical, M, dsize=img.shape[:2])

    return mask_theoretical, float(np.sum(mask_theoretical))

In [ ]:
# === Sous-fonction 3 : segmentation BiSeNet ===
def segment_with_bisenet(net, to_tensor, img, device='cpu'):
    """Run BiSeNet face parsing sur l'image et retourne le parsing argmax.

    Args:
        net: modele BiSeNet
        to_tensor: torchvision transforms compose (ToTensor + Normalize)
        img: numpy array BGR (sera converti automatiquement par to_tensor)
        device: 'cpu' ou 'cuda'

    Returns:
        parsing: numpy array HxW d'entiers (indices de classe 0-18 pour BiSeNet)
    """
    input_tensor = to_tensor(img).unsqueeze(0).to(device)
    out = net(input_tensor)[0]
    return out.squeeze(0).cpu().numpy().argmax(0)

In [ ]:
# === Sous-fonction 4 : segmentation SegFormer ===
def segment_with_segformer(seg_model, seg_processor, img, device='cpu'):
    """Run SegFormer face parsing sur l'image et retourne le parsing argmax.

    Args:
        seg_model: modele SegFormer (transformers SegformerForSemanticSegmentation)
        seg_processor: SegformerImageProcessor associe
        img: numpy array BGR
        device: 'cpu' ou 'cuda'

    Returns:
        parsing: numpy array HxW d'entiers (indices de classe 0-18 pour SegFormer)
    """
    inputs = seg_processor(images=img, return_tensors="pt").to(device)
    outputs = seg_model(**inputs)
    # Upsample les logits a la taille de l'image originale
    h, w = img.shape[:2]
    upsampled = nn.functional.interpolate(
        outputs.logits, size=(h, w), mode='bilinear', align_corners=False
    )
    return upsampled.argmax(dim=1)[0].cpu().numpy()

In [ ]:
# === Sous-fonction 5 : calcul generique des ratios hair/hat/other ===
def compute_ratios_from_parsing(parsing, mask_theoretical, total_pixels,
                                 VISIBLE_CLASSES, HAIR_CLASSES, HAT_CLASSES):
    """Calcule (hair_ratio, hat_ratio, other_ratio) a partir d'un parsing.

    Generique : fonctionne pour BiSeNet ou SegFormer en passant les bonnes listes
    de classes.

    Args:
        parsing: numpy array HxW d'indices de classe
        mask_theoretical: numpy array HxW (uint8 0/1) du mask
        total_pixels: nombre total de pixels du mask (float)
        VISIBLE_CLASSES: liste des classes 'visibles' (sera filtree)
        HAIR_CLASSES: liste des classes 'cheveux' (typiquement [17] BiSeNet, [13] SegFormer)
        HAT_CLASSES: liste des classes 'chapeau' (typiquement [18] BiSeNet, [14] SegFormer)

    Returns:
        (hair_ratio, hat_ratio, other_ratio) : 3 floats dont la somme + visible_ratio = 1.0
    """
    # Masks binaires par categorie
    visible_mask = np.isin(parsing, VISIBLE_CLASSES).astype(np.uint8)
    hair_mask = np.isin(parsing, HAIR_CLASSES).astype(np.uint8)
    hat_mask = np.isin(parsing, HAT_CLASSES).astype(np.uint8)

    # Pixels par categorie dans le mask theorique
    visible_pixels = float(np.sum(visible_mask & mask_theoretical))
    hair_pixels    = float(np.sum(hair_mask    & mask_theoretical))
    hat_pixels     = float(np.sum(hat_mask     & mask_theoretical))
    # Other = total - (visible + hair + hat). Conventionnellement clip a 0.
    other_pixels = max(0.0, total_pixels - visible_pixels - hair_pixels - hat_pixels)

    return (
        hair_pixels / total_pixels,
        hat_pixels / total_pixels,
        other_pixels / total_pixels,
        visible_pixels / total_pixels,
    )

In [ ]:
# === Sous-fonction 6 : calcul des 3 features pour detection plante v2 ===
def compute_fallback_features(parsing_b, parsing_s, mask_theoretical, total_pixels):
    """Calcule les 3 features supplementaires necessaires au fallback v2.

    Ces features sont independantes de la formule principale et servent uniquement
    a detecter quand BiSeNet ou SegFormer plante.

    Args:
        parsing_b: parsing BiSeNet (sortie de segment_with_bisenet)
        parsing_s: parsing SegFormer (sortie de segment_with_segformer)
        mask_theoretical: mask theorique (sortie de compute_theoretical_face_mask)
        total_pixels: nombre total de pixels du mask

    Returns:
        (skin_only_b_ratio, bg_b_ratio, skin_only_s_ratio) : 3 floats dans [0, 1]
    """
    skin_only_b = np.isin(parsing_b, SKIN_ONLY_B).astype(np.uint8)
    bg_b        = np.isin(parsing_b, BG_B).astype(np.uint8)
    skin_only_s = np.isin(parsing_s, SKIN_ONLY_S).astype(np.uint8)
    return (
        float(np.sum(skin_only_b & mask_theoretical)) / total_pixels,
        float(np.sum(bg_b        & mask_theoretical)) / total_pixels,
        float(np.sum(skin_only_s & mask_theoretical)) / total_pixels,
    )

In [ ]:
def compute_occlusion_score(pred_gender,visible_pixels_ratio_b,visible_pixels_ratio_s,hair_ratio_b,hair_ratio_s,hat_ratio_b,hat_ratio_s,mode="test"):

    ### Occlusion score

    occlusion_score_b = 1.0 - visible_pixels_ratio_b
    occlusion_score_s = 1.0 - visible_pixels_ratio_s   

    if mode=="test":
        if pred_gender == 'M':
            if hair_ratio_b >= 0.1:
                occlusion_score_b = np.clip(M_WEIGHTS["hair_bi"][0] * occlusion_score_b + M_WEIGHTS["hair_bi"][1],0,1)
            elif hat_ratio_b >= 0.1:
                occlusion_score_b = np.clip(M_WEIGHTS["hat_bi"][0] * occlusion_score_b + M_WEIGHTS["hat_bi"][1],0,1)
            else:
                occlusion_score_b = np.clip(M_WEIGHTS["other_bi"][0] * occlusion_score_b + M_WEIGHTS["other_bi"][1],0,1)

            if hair_ratio_s >= 0.1:
                occlusion_score_s = np.clip(M_WEIGHTS["hair_sf"][0] * occlusion_score_s + M_WEIGHTS["hair_sf"][1],0,1)
            elif hat_ratio_s >= 0.1:
                occlusion_score_s = np.clip(M_WEIGHTS["hat_sf"][0] * occlusion_score_s + M_WEIGHTS["hat_sf"][1],0,1)
            else:
                occlusion_score_s = np.clip(M_WEIGHTS["other_sf"][0] * occlusion_score_s + M_WEIGHTS["other_sf"][1],0,1)        

        elif pred_gender == 'F':
            if hair_ratio_b >= 0.1:
                occlusion_score_b = np.clip(F_WEIGHTS["hair_bi"][0] * occlusion_score_b + F_WEIGHTS["hair_bi"][1],0,1)
            elif hat_ratio_b >= 0.1:
                occlusion_score_b = np.clip(F_WEIGHTS["hat_bi"][0] * occlusion_score_b + F_WEIGHTS["hat_bi"][1],0,1)
            else:
                occlusion_score_b = np.clip(F_WEIGHTS["other_bi"][0] * occlusion_score_b + F_WEIGHTS["other_bi"][1],0,1)

            if hair_ratio_s >= 0.1:
                occlusion_score_s = np.clip(F_WEIGHTS["hair_sf"][0] * occlusion_score_s + F_WEIGHTS["hair_sf"][1],0,1)
            elif hat_ratio_s >= 0.1:
                occlusion_score_s = np.clip(F_WEIGHTS["hat_sf"][0] * occlusion_score_s + F_WEIGHTS["hat_sf"][1],0,1)
            else:
                occlusion_score_s = np.clip(F_WEIGHTS["other_sf"][0] * occlusion_score_s + F_WEIGHTS["other_sf"][1],0,1) 

    return occlusion_score_b, occlusion_score_s

In [ ]:
DEFAULT_DAMPENING_THRESHOLD = 0.35
DEFAULT_DAMPENING_SLOPE = 0.5


def damp(x, threshold, slope):
    """Piecewise linear : identite en dessous du seuil, pente reduite au-dessus."""
    if x < threshold:
        return x
    return threshold + slope * (x - threshold)

In [ ]:
def apply_dampening(hair_ratio_b, hat_ratio_b,
                        hair_ratio_s, hat_ratio_s,
                        dampening_threshold=DEFAULT_DAMPENING_THRESHOLD,
                        dampening_slope=DEFAULT_DAMPENING_SLOPE):
    """Variante de apply_v2_fallback qui ajoute un dampening sur hair et hat.

    Mecanique identique a apply_v2_fallback (meme detection plante, meme fallback)
    SAUF que hair_ratio_b, hair_ratio_s, hat_ratio_b, hat_ratio_s sont dampes
    avant utilisation dans la formule. other_ratio_b et other_ratio_s NE SONT PAS
    dampes (pas de pattern de sur-pred observe dessus).

    Args:
        ... (identiques a apply_v2_fallback) ...
        dampening_threshold (float, default 0.35) : seuil au-dessus duquel la pente
            est reduite. En dessous, comportement lineaire identique a apply_v2_fallback.
        dampening_slope (float, default 0.5) : pente au-dessus du seuil. Mettre 1.0
            pour desactiver le dampening (revient a apply_v2_fallback).

    Returns:
        float : occlusion score clip entre 0 et 1.
    """
    # Damp hair et hat (BiSeNet et SegFormer). other_b et other_s INCHANGES.
    hair_b_d = damp(hair_ratio_b, dampening_threshold, dampening_slope)
    hat_b_d  = damp(hat_ratio_b,  dampening_threshold, dampening_slope)
    hair_s_d = damp(hair_ratio_s, dampening_threshold, dampening_slope)
    hat_s_d  = damp(hat_ratio_s,  dampening_threshold, dampening_slope)

    # Le reste : identique a apply_v2_fallback mais avec features dampees
    return hair_b_d, hat_b_d, hair_s_d, hat_s_d

In [ ]:
def apply_fallback(occlusion_score_b,occlusion_score_s,bg_b_ratio, skin_only_s_ratio):
    """Calcule le score d'occlusion avec gestion automatique des plantages.

    Cette fonction remplace le bloc `if pred_gender == 'M' / elif 'F'` du notebook.
    Elle se comporte EXACTEMENT comme la formule actuelle quand aucun modele ne
    plante (cas normal, ~98% des images). Quand un modele plante, elle bascule
    intelligemment sur l'autre.

    Args:
        pred_gender (str): 'M' ou 'F', sortie de face.sex (InsightFace).
        hair_ratio_b, hat_ratio_b, other_ratio_b (float): ratios BiSeNet de Julien.
        hair_ratio_s, hat_ratio_s, other_ratio_s (float): ratios SegFormer de Julien.
        skin_only_b_ratio (float): NOUVEAU. Fraction de skin pur (sans bg, sans eye_g,
                                   avec ears) detectee par BiSeNet dans le mask.
                                   Utilise pour diagnostic/debug, pas dans la decision.
        bg_b_ratio (float):        NOUVEAU. Fraction de background detectee par BiSeNet
                                   dans le mask. Si > 0.70 -> BiSeNet plante.
        skin_only_s_ratio (float): NOUVEAU. Fraction de skin pur (sans bg, sans eye_g,
                                   avec ears) detectee par SegFormer dans le mask.
                                   Si < 0.30 -> SegFormer plante.
        M_WEIGHTS, F_WEIGHTS (dict): tes coefficients du notebook (inchanges).

    Returns:
        float: score d'occlusion clip entre 0 et 1 (= ton ancien occlusion_score).
    """

    # ------ ETAPE 2 : detection plantage (independante du genre) ------
    # bg_b_ratio > 0.70 : BiSeNet a etiquette ~tout le visage en background
    # skin_only_s_ratio < 0.30 : SegFormer ne voit quasi aucune peau
    bisenet_error = bg_b_ratio > BI_ERROR_THRESHOLD
    segformer_error = skin_only_s_ratio < SF_ERROR_THRESHOLD

    # ------ ETAPE 3 : choix de la formule ------
    if segformer_error and not bisenet_error:
        # CAS 1 : SegFormer plante mais BiSeNet OK
        # -> on calcule pred avec UNIQUEMENT la moitie BiSeNet (3 termes au lieu de 6)
        # -> on ignore les ratios SegFormer (ils sont absurdes : other_ratio_s ~ 0.95)
        score = occlusion_score_b

    elif bisenet_error and not segformer_error:
        # CAS 2 : BiSeNet plante mais SegFormer OK
        # -> on calcule pred avec UNIQUEMENT la moitie SegFormer
        # -> on ignore les ratios BiSeNet
        score = occlusion_score_s

    elif bisenet_error and segformer_error:
        # CAS 3 : les 2 plantent -> on prend "le moins pire"
        # On mesure le degre de plantage de chaque modele (haut = plus plante) :
        #   - BiSeNet : bg_b_ratio (plus c'est haut, plus BiSeNet a sature en bg)
        #   - SegFormer : 1 - skin_only_s_ratio (plus c'est haut, moins SegFormer voit de peau)
        # On bascule sur le modele dont le score de plantage est le plus bas.
        error_score_bisenet = bg_b_ratio
        error_score_segformer = 1.0 - skin_only_s_ratio
        if error_score_bisenet < error_score_segformer:
            # BiSeNet est "moins pire" -> sa contribution est plus fiable
            score = occlusion_score_b
        else:
            # SegFormer est "moins pire" -> sa contribution est plus fiable
            score = occlusion_score_s

    else:
        # CAS 4 (le plus frequent, ~98% des images) : aucun modele ne plante
        # -> on utilise la formule complete habituelle, identique a ton notebook
        score = 0.5*occlusion_score_b + 0.5*occlusion_score_s
        
    # ------ ETAPE 4 : clip final entre 0 et 1 (identique a ton notebook) ------
    return score


In [ ]:
# === Orchestrateur : occlusion_computation (appelle toutes les sous-fonctions) ===
def occlusion_computation(app, img, display_results=False, mode="test"):
    """Pipeline complete : detecte le visage, segmente, calcule les ratios,
    applique le v2 fallback + dampening, retourne le score d'occlusion.

    Signature compatible avec les cellules suivantes du notebook qui appellent
    occlusion_computation(app, img) pour la generation des resultats val / test.

    Args:
        app: instance FaceAnalysis InsightFace
        img: numpy array BGR
        display_results: si True, affiche les overlays (debug visuel)

    Returns:
        occlusion_score : float dans [0, 1] (= ton ancien y_pred)
    """
    # Resize unique a 512x512 pour unifier le pipeline (logique Julien)
    img = cv2.resize(img, (512, 512), interpolation=cv2.INTER_CUBIC)

    # === Etape 1 : detection visage + genre ===
    bbox, pred_gender = detect_face_and_gender(app, img)
    if bbox is None:
        return 0.0

    # === Etape 2 : mask theorique 3D avec correction Julien ===
    mask_theoretical, total_pixels = compute_theoretical_face_mask(tddfa, img, bbox)
    if total_pixels == 0:
        return 0.0

    # === Etape 3 : segmentations BiSeNet + SegFormer ===
    parsing_b = segment_with_bisenet(net, to_tensor, img, device=device)
    parsing_s = segment_with_segformer(seg_model, seg_processor, img, device=device)

    # === Etape 4 : ratios hair/hat/other/visible pour chaque modele ===
    hair_ratio_b, hat_ratio_b, other_ratio_b, visible_pixels_ratio_b = compute_ratios_from_parsing(
        parsing_b, mask_theoretical, total_pixels,
        VISIBLE_FACE_CLASSES_B, HAIR_B, HAT_B,
    )
    hair_ratio_s, hat_ratio_s, other_ratio_s, visible_pixels_ratio_s = compute_ratios_from_parsing(
        parsing_s, mask_theoretical, total_pixels,
        VISIBLE_FACE_CLASSES_S, HAIR_S, HAT_S,
    )

    # === Etape 5 : features de detection d'erreur segmentation (pour le fallback) ===
    skin_only_b_ratio, bg_b_ratio, skin_only_s_ratio = compute_fallback_features(
        parsing_b, parsing_s, mask_theoretical, total_pixels,
    )

    # === Etape 6 : dampening ===
    hair_ratio_bd, hat_ratio_bd, hair_ratio_sd, hat_ratio_sd = apply_dampening(hair_ratio_b, hat_ratio_b, hair_ratio_s, hat_ratio_s)

    # === Etape 7 : occlusion score ===
    occlusion_score_b, occlusion_score_s = compute_occlusion_score(pred_gender,visible_pixels_ratio_b,visible_pixels_ratio_s,hair_ratio_bd,hair_ratio_sd,hat_ratio_bd,hat_ratio_sd,mode)

    # === Etape 8 : calcul du score avec fallback ===
    occlusion_score = apply_fallback(occlusion_score_b,occlusion_score_s,bg_b_ratio, skin_only_s_ratio)

    # === Visualisation optionnelle (debug) ===
    if display_results:
        visible_skin_mask_b = np.isin(parsing_b, VISIBLE_FACE_CLASSES_B).astype(np.uint8)
        visible_skin_mask_s = np.isin(parsing_s, VISIBLE_FACE_CLASSES_S).astype(np.uint8)
        vis1 = overlay_mask(img, mask_theoretical, color=(0, 255, 0), alpha=0.35)
        vis2 = overlay_mask(vis1, visible_skin_mask_b, color=(255, 0, 0), alpha=0.35)
        vis3 = overlay_mask(vis2, visible_skin_mask_s, color=(255, 0, 0), alpha=0.35)
        plt.figure(figsize=(8, 8))
        plt.imshow(cv2.cvtColor(vis3, cv2.COLOR_BGR2RGB))
        plt.axis("off")
        plt.title(
            f"{filename}\n"
            f"GT occlusion={occlusion:.3f} | {gender} - {pred_gender} | Pred occlusion={occlusion_score:.3f}"
        )
        plt.show()

    return occlusion_score


# Generate test predictions

### Make predictions on test dataset

In [ ]:
test_set = Dataset(df_test1, image_dir, training=False)

params_val = {'batch_size': 1,
          'shuffle': False,
          'num_workers': 0}

test_generator = torch.utils.data.DataLoader(test_set, **params_val)

In [ ]:
results_list = []
with torch.inference_mode():
    for batch_idx, (X, y, filename) in tqdm(enumerate(test_generator), total=len(test_generator)):
        for i in range(len(X)):
            img = cv2.imread(os.path.join(image_dir, filename[i]))
            y_pred = occlusion_computation(app,img)
            results_list.append({'filename': filename[i],
                                 'FaceOcclusion': float(y_pred),
                                 'row_id': int(y[i]),
                                 })
results_df = pd.DataFrame(results_list)

In [ ]:
results_df.head()

### Export predictions
Note: We need to add a dummy 'gender' column for the hfactory upload.

In [ ]:
results_df['gender'] = 'x'
results_df.to_csv("test1_predictions.csv", sep=',', index=False)